#1. Basic Tasks 

##1. Create a catalog cyntexa_dev and a schema sales within it. 

In [0]:
CREATE CATALOG cyntexa_dev;
create schema cyntexa_dev.sales;

##2. Create a managed table sales.orders_raw with at least 5 columns and insert 10 sample rows. 

In [0]:
create or replace table cyntexa_dev.sales.orders_raw(order_id int, order_date date, order_customer_id int, order_status string,order_amount double);
insert into cyntexa_dev.sales.orders_raw values (1, '2026-01-01',10001, 'placed', 100.0), (2, '2026-01-02', 10002, 'shipped', 200.0), (3, '2026-01-03', 10003,'completed', 300.0), (4, '2026-01-04', 10004, 'placed', 400.0), (5,'2026-01-05', 10005, 'shipped', 500.0), (6, '2026-01-06', 10006, 'completed', 600.0), (7, '2026-01-07', 10007, 'placed', 700.0), (8, '2026-01-08', 10008, 'shipped', 800.0), (9, '2026-01-09', 10009, 'completed', 900.0), (10, '2026-01-10', 10010, 'placed', 1000.0);
select * from cyntexa_dev.sales.orders_raw


##3.Create a view sales.orders_view that selects only completed orders.

In [0]:
create or replace view cyntexa_dev.sales.orders_view as select * from cyntexa_dev.sales.orders_raw where order_status = 'completed';
select * from cyntexa_dev.sales.orders_view

##4. (Data Analyst) Explore samples.bakehouse or samples.tpch and run 3 exploratory SELECT queries. 

In [0]:
-- Exploratory Query 1: Examine the customer table structure and sample data
SELECT * FROM samples.tpch.customer LIMIT 10;

-- Exploratory Query 2: Analyze orders by status
SELECT o_orderstatus, COUNT(*) as order_count, SUM(o_totalprice) as total_revenue
FROM samples.tpch.orders
GROUP BY o_orderstatus
ORDER BY total_revenue DESC;

-- Exploratory Query 3: Top 10 customers by total purchase amount
SELECT 
  c.c_custkey,
  c.c_name,
  c.c_nationkey,
  SUM(o.o_totalprice) as total_spent,
  COUNT(o.o_orderkey) as order_count
FROM samples.tpch.customer c
JOIN samples.tpch.orders o ON c.c_custkey = o.o_custkey
GROUP BY c.c_custkey, c.c_name, c.c_nationkey
ORDER BY total_spent DESC
LIMIT 10;


In [0]:
-- Create a SQL UDF that masks the last 4 characters
CREATE OR REPLACE FUNCTION mask_last_4(input STRING)
RETURNS STRING
RETURN CONCAT(REPEAT('*', GREATEST(0, LENGTH(input) - 4)), right(input, 4));

-- Apply the UDF to query orders_raw
SELECT
  mask_last_4(CAST(order_customer_id AS STRING)) AS masked_order__customer_id,
  order_date,
  order_status,
  order_amount
FROM
  cyntexa_dev.sales.orders_raw;

In [0]:
-- For comparison purposes, create a second MANAGED table
-- (External tables require a registered external location with s3:// path on AWS)
-- This demonstrates the difference between two managed tables with different purposes
CREATE OR REPLACE TABLE cyntexa_dev.sales.orders_external (
  order_id INT,
  order_date DATE,
  order_customer_id INT,
  order_status STRING,
  order_amount DOUBLE
);

-- Insert some sample data
INSERT INTO cyntexa_dev.sales.orders_external VALUES 
  (101, '2026-02-01', 20001, 'pending', 150.0),
  (102, '2026-02-02', 20002, 'completed', 250.0);

-- Compare the two tables
-- orders_raw: Original managed table created in Cell 5
-- orders_external: Demonstrates table structure and operations (managed for this demo)
DESCRIBE EXTENDED cyntexa_dev.sales.orders_raw;

In [0]:
--Without s3 bucket location access we cannot create table in s3 in free edition
-- 1. Create a managed table
CREATE OR REPLACE TABLE demo_managed_table (
    id INT,
    name STRING
);

-- 2. Insert dummy data
INSERT INTO demo_managed_table VALUES (1, 'Alice'), (2, 'Bob');

-- 3. Create a volume first (required for external table location)
CREATE VOLUME IF NOT EXISTS cyntexa_dev.sales.external_data;

-- 4. Create an external table using the volume path
CREATE OR REPLACE TABLE demo_external_table (
    id INT,
    name STRING
)
USING DELTA
LOCATION '/Volumes/cyntexa_dev/sales/external_data/demo_external';

-- 5. Insert dummy data
INSERT INTO demo_external_table VALUES (10, 'Charlie'), (20, 'David');

-- Inspect the Managed Table metadata
DESCRIBE EXTENDED demo_managed_table;

-- Inspect the External Table metadata
DESCRIBE EXTENDED demo_external_table;


[RequestId=7df8a9e0-5972-4ba8-a7a9-609a30805a1b ErrorClass=INVALID_PARAMETER_VALUE.INVALID_PARAMETER_VALUE] Missing cloud file system scheme

##7.(Data Analyst) Build a second view joining orders_view with a customers table/view and calculate total spend per customer. 

In [0]:
-- First, create a sample customers table to join with
CREATE OR REPLACE TABLE cyntexa_dev.sales.customers (
  customer_id INT,
  customer_name STRING,
  customer_email STRING
);

-- Insert sample customer data
INSERT INTO cyntexa_dev.sales.customers VALUES
  (10001, 'John Doe', 'john@example.com'),
  (10002, 'Jane Smith', 'jane@example.com'),
  (10003, 'Bob Johnson', 'bob@example.com'),
  (10004, 'Alice Williams', 'alice@example.com'),
  (10005, 'Charlie Brown', 'charlie@example.com'),
  (10006, 'Diana Prince', 'diana@example.com'),
  (10007, 'Eve Davis', 'eve@example.com'),
  (10008, 'Frank Miller', 'frank@example.com'),
  (10009, 'Grace Lee', 'grace@example.com'),
  (10010, 'Henry Wilson', 'henry@example.com');

-- Create a view joining orders_view with customers and calculate total spend per customer
CREATE OR REPLACE VIEW cyntexa_dev.sales.customer_total_spend AS
SELECT 
  c.customer_id,
  c.customer_name,
  c.customer_email,
  o.order_status,
  COUNT(o.order_id) AS total_orders,
  SUM(o.order_amount) AS total_spend
FROM cyntexa_dev.sales.customers c
LEFT JOIN cyntexa_dev.sales.orders_view o 
  ON c.customer_id = o.order_customer_id
GROUP BY c.customer_id, c.customer_name, c.customer_email, o.order_status
ORDER BY total_spend DESC;

-- Query the new view
SELECT * FROM cyntexa_dev.sales.customer_total_spend;

#3. Advanced Tasks 

##8. Design a full three-level namespace plan for Cyntexa (catalogs for dev/staging/prod, schemas per business domain) and justify the structure in a short writeup. 


## Three-Level Namespace Plan for Cyntexa

### Namespace Structure

#### **Catalogs (Environment-based)**
- **`cyntexa_dev`** - Development environment for active development and testing
- **`cyntexa_staging`** - Staging/UAT environment for pre-production validation
- **`cyntexa_prod`** - Production environment for live business operations

#### **Schemas (Business Domain-based)**
Each catalog contains the same schema structure:

1. **`sales`** - Sales transactions, orders, customers, revenue data
2. **`marketing`** - Campaign data, leads, customer engagement, analytics
3. **`finance`** - Financial transactions, invoicing, payments, accounting
4. **`operations`** - Supply chain, inventory, logistics, fulfillment
5. **`hr`** - Employee data, payroll, performance, recruitment
6. **`analytics`** - Cross-domain aggregations, KPIs, business intelligence models
7. **`raw`** - Landing zone for ingested data before transformation
8. **`archive`** - Historical data and compliance records

#### **Tables/Views (Entity-based)**
Examples within each schema:
- `cyntexa_prod.sales.orders_raw`
- `cyntexa_prod.sales.orders_clean`
- `cyntexa_prod.sales.customer_total_spend`
- `cyntexa_staging.marketing.campaigns`
- `cyntexa_dev.analytics.daily_revenue_kpi`

---

### Justification
- **Isolation**: Each environment (dev/staging/prod) has complete data isolation, preventing accidental cross-environment impacts
- **Access Control**: Unity Catalog allows granular permissions at the catalog level - developers get full access to `cyntexa_dev`, read-only to `cyntexa_staging`, and restricted access to `cyntexa_prod`
- **Deployment Pipeline**: Promotes a clear CI/CD pattern where code and schema changes flow dev → staging → prod
- **Cost Management**: Easy to track compute and storage costs per environment


---



##9. Write a data-masking strategy: which columns need masking, which role tiers should see unmasked data, and how Unity Catalog permissions would enforce it (this connects forward to Day 8 governance). 

## Data Masking Strategy for Cyntexa

### Data Masking Strategy
####1. Columns Needing Masking (by Schema)

`sales` (`orders_raw`, `customer_total_spend`, `customers`, etc.)

- Direct identifiers: `customer_name`, `email`, `phone`, `billing_address`, `shipping_address`
- Financial: `card_last4`/`payment_token`, `total_spend` amounts

`marketing` (`campaigns`, `leads`)
- `lead_email`, `lead_phone`, `contact_name`
- `device_id`, `ip_address` 
`finance` (invoicing, payments, accounting)

- `bank_account_number`, `routing_number`, `card_number`, `tax_id`/`SSN`, `salary_data` 
- Invoice-level `customer_contact_info`

`operations` (supply chain, logistics)

- Generally low PII — mostly `vendor_contact`, `driver_name`/`driver_id` if fulfillment data includes personnel

`hr` — highest sensitivity schema overall

- `employee_ssn`, `salary`, `bank_details`, `home_address`, `performance_reviews`, `date_of_birth`, `emergency_contact`

`analytics` (`daily_revenue_kpi`)

- Should ideally be pre-aggregated

`raw` — landing zone

- Treat as highest-risk by default: everything inbound is unclassified until tagged, so default-deny/mask-everything until classification runs

`archive`

- Same sensitivity as source schema, but often needs stricter access since it's compliance/audit-oriented (retention data), even though access frequency is lower
---
**Key principle:** `cyntexa_prod` is where masking really matters. `cyntexa_dev` should ideally run on sample data entirely, which removes the need for runtime masking in dev — this is often cheaper and safer than masking real data everywhere. `cyntexa_staging` sits in between: if staging is refreshed from prod, it needs the same masking as prod.

**Domain-based restriction is orthogonal to the tier table above —** e.g., a sales analyst should not see unmasked `hr.employee_ssn` regardless of environment.

###2. Unity Catalog Enforcement — Structural Mapping

####Group design mirrors the namespace:

- Domain groups: `sales_analysts`, `finance_analysts`, `hr_admins`, etc.

####Catalog-level grants (environment boundary):

- `USE CATALOG` on `cyntexa_prod` restricted to a small set — most analysts shouldn't even browse prod directly if staging/analytics serves their need.
- `cyntexa_dev` can have broader `USE CATALOG` grants since data is sample

####Schema-level grants (domain boundary):

- `GRANT SELECT ON SCHEMA cyntexa_prod.hr TO hr_admins` — no cross-domain grants by default (a marketing analyst gets no grant on `hr` or `finance` schemas at all, not even masked)

#####Column-level masking functions (tier boundary within a domain):

- Attached per sensitive column, e.g. on `cyntexa_prod.sales.customer_total_spend.email`:
 - Function checks `is_account_group_member('tier2_engineer')` OR `is_account_group_member('tier3_compliance')` → return raw value; else → return masked value
- Same function definition can be reused across `sales`, `marketing`, `finance` schemas since the tier-check logic is identical — only the column and domain group differ

**Row filters (optional, complementary):**

- E.g., regional sales reps only see rows where `region = current_user_region()`, layered under the same column masks


**Tagging strategy tying it together:**

- Apply Unity Catalog column tags (e.g., `pii_level=high`) at the `raw`→domain promotion step, and write masking policies against the tag rather than hardcoding column names per table — this scales naturally as you add new tables under `sales`, `marketing`, etc. without rewriting masking rules each time




##10. (Data Analyst) Using samples.tpch, write a query with at least one CTE and one window function to produce a 'top 5 customers by revenue per region' report. 

In [0]:
WITH customer_revenue AS (
  SELECT 
    r.r_name AS region_name,
    c.c_custkey,
    c.c_name,
    SUM(o.o_totalprice) AS total_revenue
  FROM samples.tpch.orders o
  INNER JOIN samples.tpch.customer c ON o.o_custkey = c.c_custkey
  INNER JOIN samples.tpch.nation n ON c.c_nationkey = n.n_nationkey
  INNER JOIN samples.tpch.region r ON n.n_regionkey = r.r_regionkey
  GROUP BY r.r_name, c.c_custkey, c.c_name
),
ranked_customers AS (
  SELECT 
    region_name,
    c_custkey,
    c_name,
    total_revenue,
    ROW_NUMBER() OVER (PARTITION BY region_name ORDER BY total_revenue DESC) AS rank_in_region
  FROM customer_revenue
)
SELECT 
  region_name,
  c_custkey,
  c_name,
  total_revenue,
  rank_in_region
FROM ranked_customers
WHERE rank_in_region <= 5
ORDER BY region_name, rank_in_region


